# Lesson 2 — Read and Plot Your NeuraDock Data

**Required · real-recording replay · about 60 minutes**

Lesson 1 built an explanatory simulation. Here we open a **real recording**:

**Text file → seven-channel array → relative time → raw waveforms.**

By the end, you can read a file, explain `data[channel, sample]`, and plot a
chosen channel and time interval. You will generate three figures yourself.
No filtering, normalization, PSD, automated quality score, or classification
is performed. Reading a file successfully does not establish signal quality.

**VS Code:** select the course Python kernel, then choose **Run All**.
A real file is required; this lesson never substitutes synthetic data.
Run the example unchanged first. Later change only the three viewing controls.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys

locations = (Path.cwd(), Path.cwd() / "neuradock-eeg-101", *Path.cwd().parents)
ROOT = next((p for p in locations if (p / "src" / "neuradock_eeg101").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open the neuradock-eeg-101 project folder in VS Code before running this notebook.")
sys.path.insert(0, str(ROOT / "src"))
IN_NOTEBOOK = "ipykernel" in sys.modules
if IN_NOTEBOOK:
    get_ipython().run_line_magic("matplotlib", "inline")
else:
    os.environ.setdefault("MPLBACKEND", "Agg")
import numpy as np
import matplotlib.pyplot as plt
from neuradock_eeg101.reading import inspect_timing, describe_p_field

OUTPUT_DIR = ROOT / "outputs" / "notebooks" / "lesson-02"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES = []
plt.rcParams.update({"font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False})

def show_and_save(fig, filename):
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=150, bbox_inches="tight", facecolor="white")
    FIGURES.append(filename)
    if IN_NOTEBOOK:
        plt.show()
    plt.close(fig)


## 1. Open a real file, not a pre-made figure · 10 min

The public teaching-data location is `data/teaching/lesson-02/recording-01.txt`.
See that directory's README for availability and permission status. In the
author's workspace, the original `../S2/S04_01.txt` can be read directly without
copying it into the repository. Both choices are real recordings.

For another file, replace `DATA_PATH` below with its path (relative paths are
relative to the repository), or set `NEURADOCK_LESSON02_FILE` before starting
the kernel. A missing file raises a clear error; there is **no synthetic fallback**.
Keep private recordings and executed notebook outputs out of Git.


In [ ]:
TEACHING_FILE = ROOT / "data" / "teaching" / "lesson-02" / "recording-01.txt"
LOCAL_ORIGINAL = ROOT.parent / "S2" / "S04_01.txt"
default_file = TEACHING_FILE if TEACHING_FILE.is_file() else LOCAL_ORIGINAL
DATA_PATH = Path(os.environ.get("NEURADOCK_LESSON02_FILE", str(default_file)))
if not DATA_PATH.is_absolute():
    DATA_PATH = ROOT / DATA_PATH
DATA_PATH = DATA_PATH.resolve()
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "Lesson 2 requires a real recording. Place the authorized teaching file at "
        "data/teaching/lesson-02/recording-01.txt, or set DATA_PATH to your own real file. "
        "No synthetic substitute will be used.")
if (ROOT / "data" / "synthetic").resolve() in DATA_PATH.parents:
    raise ValueError("Lesson 2 requires real data, not a bundled synthetic recording.")
SOURCE_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
if os.environ.get("NEURADOCK_LESSON02_VERIFY") == "1":
    OUTPUT_DIR = OUTPUT_DIR / "verification" / SOURCE_SHA256[:12]
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Acquisition settings, NOT parameters to tune for a nicer-looking waveform.
FS = 250.0  # Declared in the supplied S2 notebook; verify for any different file.
CHANNEL_NAMES = [f"Ch{i + 1}" for i in range(7)]  # Column labels, not scalp locations.
AMPLITUDE_LABEL = "Recorded amplitude (unit unconfirmed)"
TIMESTAMP_FORMAT = "clock_hms_ms"  # S2 uses HH:MM:SS.mmm; alternatives: seconds, unknown.

print("Source: user-supplied real recording (local replay)")
print("File:", DATA_PATH.name)
print("Nominal sample rate:", FS, "Hz; check acquisition documentation")
print("Original S2 notes assume microvolts; this file header does not confirm calibration.")
print("No automatic electrode names or experimental-event labels are assigned.")
print("Real-data provenance comes from the data owner, not from the text layout itself.")


A text header describes the **columns**, not the physiological interpretation:

`HEADER_DEF,T,P,C,C,C,C,C,C,C,0`

`T` is a recorded time field; keep the original text. `P` is a raw metadata
field whose meaning must be established for this file. Each `C` contains an
EEG channel value. `0` marks a reserved column, not an eighth EEG channel.
A seven-`C` row contains one sample; a 35-`C` row contains five successive
seven-channel samples. We focus on the supplied one-sample-per-row file.


In [ ]:
with DATA_PATH.open(encoding="utf-8-sig") as stream:
    header_line = stream.readline().strip()
    preview_rows = [stream.readline().strip() for _ in range(2)]
header_parts = header_line.split(",")[1:]
channel_indices = [i for i, field in enumerate(header_parts) if field.strip() == "C"]
print("Header:", header_line)
print("EEG columns (zero-based):", channel_indices)
print("Number of EEG fields:", len(channel_indices))
print("Fields in each preview row:", [len(row.split(",")) for row in preview_rows if row])
print("The reader below will validate the complete layout before extracting values.")


## 2. Turn the text into an array · 15 min

The function below preserves the core of the supplied
`S2/S2_CP_evaluations_share.ipynb` reader:

1. Find the `C` columns in the header.
2. Read seven floats per sample, in their original order.
3. Transpose the result into **channels × samples**.

This is an adaptation, not a verbatim copy: it adds strict format and finite-value
checks, preserves `T` and `P`, and rejects malformed data instead of treating
every failed row as an event. The small support helpers perform validation;
the extraction and transpose are visible here. No signal values are rescaled.


In [ ]:
from neuradock_eeg101.reading import _clock_hms_ms_to_seconds, _malformed, _strict_error, _validated_header

def data_reader(file_path: str | Path, *, strict: bool = True,
                event_prefix: str | None = None) -> tuple[np.ndarray, dict[str, object]]:
    """Read the exact public packet layout without guessing labels, units, or events."""
    path = Path(file_path)
    with path.open(encoding="utf-8-sig") as stream:
        lines = stream.read().splitlines()
    if not lines:
        raise ValueError("The recording is empty.")
    header, channel_indices, samples_per_packet = _validated_header(lines[0])
    if event_prefix is not None and not event_prefix.strip():
        raise ValueError("event_prefix must contain a visible, explicit prefix")
    expected_fields = len(header) - 1
    samples, raw_timestamps, p_fields = [], [], []
    sample_packet_indices, packet_line_numbers = [], []
    malformed_rows, events = [], []
    for line_number, line in enumerate(lines[1:], start=2):
        stripped = line.strip()
        if event_prefix is not None and stripped.startswith(event_prefix):
            label = stripped[len(event_prefix):].strip()
            if label:
                events.append({"sample_index": len(samples), "label": label,
                               "line_number": line_number})
            else:
                malformed_rows.append(_malformed(line_number, "empty explicit event label"))
            continue
        fields = [field.strip() for field in stripped.split(",")]
        if not stripped or len(fields) != expected_fields:
            malformed_rows.append(_malformed(
                line_number, f"expected {expected_fields} fields, received {len(fields)}"))
            continue
        try:
            packet = [[float(fields[index]) for index in
                       channel_indices[group * 7:(group + 1) * 7]]
                      for group in range(samples_per_packet)]
        except ValueError:
            malformed_rows.append(_malformed(line_number, "EEG field is not numeric"))
            continue
        if not np.isfinite(packet).all():
            malformed_rows.append(_malformed(line_number, "EEG field is not finite"))
            continue
        packet_index = len(raw_timestamps)
        samples.extend(packet)
        sample_packet_indices.extend([packet_index] * samples_per_packet)
        raw_timestamps.append(fields[0])
        p_fields.append(fields[1])
        packet_line_numbers.append(line_number)
    if malformed_rows and strict:
        raise _strict_error(malformed_rows)
    if not samples:
        raise ValueError("No valid seven-channel samples were found.")
    data = np.asarray(samples, dtype=float).T
    info = {
        "header": header, "channel_indices": channel_indices,
        "samples_per_packet": samples_per_packet,
        "raw_timestamps": raw_timestamps, "p_fields": p_fields,
        "packet_count": len(raw_timestamps), "malformed_rows": malformed_rows,
        "events": events, "sample_packet_indices": sample_packet_indices,
        "packet_line_numbers": packet_line_numbers,
    }
    return data, info


In [ ]:
data, info = data_reader(DATA_PATH)
n_channels, n_samples = data.shape
assert n_channels == 7 and np.isfinite(data).all()
print("Array shape:", data.shape, "= channels x samples")
print("data[0] is the first channel; data[:, 0] is the first seven-channel sample.")
print("Packets:", info["packet_count"], "| Samples per packet:", info["samples_per_packet"])
print("Malformed rows:", len(info["malformed_rows"]))
print("No rows were silently dropped; no experimental events were inferred from P.")


## 3. Give the samples a relative time axis · 10 min

At the **declared** 250 Hz sampling rate, the nominal interval is
`1 / 250 = 0.004 s = 4 ms`. Sample index 0 has relative time 0; index 250 has
relative time 1 second. `N / FS` describes the nominal sample duration;
the last plotted sample is at `(N - 1) / FS`.

This uniform time axis is a plotting convention based on the declared rate,
**not a reconstruction of exact acquisition timing**. Keep the recorded clock
separate. Duplicated clock values or large jumps require investigation, not
silent interpolation. A missing packet cannot be repaired by drawing a smooth line.


In [ ]:
if not np.isfinite(FS) or FS <= 0:
    raise ValueError("FS must be the positive sampling rate from acquisition documentation.")
time_s = np.arange(n_samples) / FS
print("First five sample indices:", np.arange(min(5, n_samples)))
print("First five relative times (s):", time_s[:5])
print(f"Nominal sample interval: {1000 / FS:.1f} ms")
print(f"Nominal sample duration: {n_samples / FS:.3f} s")
print(f"Last sample relative time: {time_s[-1]:.3f} s")


### A short metadata check — clocks and counters are not events

The supplied S2 files contain clock strings, not floating-point seconds.
We parse them explicitly for a **packet-clock summary**, retaining the originals
in `info["raw_timestamps"]`. No device-clock timestamps are interpolated for
five-sample rows; plotting uses the separate nominal sample-index/FS axis.
Use `TIMESTAMP_FORMAT = "unknown"` when the format/unit has not been established.

In the two supplied files, `P` increments modulo 256. That supports a
**counter-like pattern**, not a stimulus or task label. The helper reports the
pattern without creating experimental events. Event timing needs a documented
event field or a separate event log; it is outside this first reading lesson.


In [ ]:
timing = inspect_timing(info["raw_timestamps"], timestamp_format=TIMESTAMP_FORMAT)
p_summary = describe_p_field(info["p_fields"])
print("Clock format:", timing["timestamp_format"], "| Status:", timing["status"])
print("Parsed packet clocks:", timing["valid_count"], "/", timing["packet_count"])
print("Median packet interval (s):", timing["median_packet_interval_s"])
print("Maximum packet interval (s):", timing["maximum_packet_interval_s"])
print("Repeated/backward clock intervals:", timing["nonpositive_interval_count"])
if timing["nonpositive_interval_count"]:
    print("Timing warning: use the nominal plot axis with caution; it does not repair clock irregularities.")
print("P-field evidence:", p_summary["interpretation"])
print("Experimental events inferred:", p_summary["experimental_events_inferred"])
print("The complete aggregate diagnostics will be saved with the final summary.")


## 4. Plot a channel, then all seven · 20 min

Set the three **viewing controls** below. `CHANNEL_INDEX = 0` means the first
data column (`Ch1`). Do not infer its scalp position from this number.
After changing a control, rerun this cell and the three plotting cells below it.

The original S2 notebook labels its data in µV, but the header alone does not
establish the unit. After confirming the acquisition/export calibration, you
may change `AMPLITUDE_LABEL` to `Amplitude (µV)`. Changing a label never converts
values. These plots preserve raw offsets and amplitudes; they are not cleaned EEG.


In [ ]:
CHANNEL_INDEX = 0  # Choose 0, 1, ..., 6.
START_S = 0.0     # Start of the displayed interval, on the nominal time axis.
WINDOW_S = 5.0    # Length of the displayed interval, in seconds.

if not isinstance(CHANNEL_INDEX, (int, np.integer)) or not 0 <= CHANNEL_INDEX < n_channels:
    raise ValueError("CHANNEL_INDEX must be an integer from 0 to 6.")
if not np.isfinite([START_S, WINDOW_S]).all() or START_S < 0 or WINDOW_S <= 0:
    raise ValueError("Use a nonnegative START_S and a positive WINDOW_S.")
start = int(round(START_S * FS))
stop = min(start + int(round(WINDOW_S * FS)), n_samples)
if start >= n_samples or stop - start < 2:
    raise ValueError("Choose a window inside the recording containing at least two samples.")
view = slice(start, stop)
view_time = time_s[view]
if stop - start < int(round(WINDOW_S * FS)):
    print("The requested window was clipped at the end of the recording.")
print(f"Viewing samples {start} to {stop - 1}, inclusive; raw data are unchanged.")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(view_time, data[CHANNEL_INDEX, view], color="#176B87", lw=0.9)
ax.set(xlabel="Nominal relative time (s)", ylabel=AMPLITUDE_LABEL,
       title=f"Real recording — {CHANNEL_NAMES[CHANNEL_INDEX]} — raw, unfiltered")
ax.ticklabel_format(axis="y", style="plain", useOffset=False)
ax.grid(alpha=0.2)
show_and_save(fig, "01-single-channel.png")


**Observe:** does the baseline sit near zero? Are there slow shifts or sudden
changes? Describe what you can see without diagnosing its source. A large
offset is still part of the recorded values; this lesson does not remove it.


In [ ]:
fig, axes = plt.subplots(7, 1, figsize=(11, 10), sharex=True)
for channel, ax in enumerate(axes):
    ax.plot(view_time, data[channel, view], color="#176B87", lw=0.8)
    ax.set_ylabel(CHANNEL_NAMES[channel])
    ax.ticklabel_format(axis="y", style="plain", useOffset=False)
    ax.grid(alpha=0.2)
axes[0].set_title("Real recording — seven raw channels; each panel has its own y-scale")
axes[-1].set_xlabel("Nominal relative time (s)")
fig.supylabel(AMPLITUDE_LABEL)
show_and_save(fig, "02-seven-channels.png")


**Observe:** do any changes occur at similar times across channels?
Each panel uses its own labeled y-scale so raw offsets remain visible.
Compare numerical axes, **not visual trace heights**, when comparing amplitudes.
Similar-looking channels alone do not prove a common neural source.


In [ ]:
zoom_stop = min(start + max(2, int(round(0.1 * FS))), stop)
zoom = slice(start, zoom_stop)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(time_s[zoom], data[CHANNEL_INDEX, zoom], "o-", color="#C26630",
        markersize=5, lw=1, label="One dot per recorded sample")
ax.set(xlabel="Nominal relative time (s)", ylabel=AMPLITUDE_LABEL,
       title=f"{CHANNEL_NAMES[CHANNEL_INDEX]} — first {zoom_stop - start} samples of this view")
ax.ticklabel_format(axis="both", style="plain", useOffset=False)
ax.legend(frameon=False)
ax.grid(alpha=0.2)
show_and_save(fig, "03-sample-points.png")


## 5. Read another part of the recording · 5 min

1. Change `CHANNEL_INDEX` from 0 to 6. Which data row is plotted now?
2. Change `START_S` to 10 and `WINDOW_S` to 2 (if your file is long enough).
   Rerun the controls and three plotting cells. How many samples are displayed?
3. In the zoomed plot, what is the nominal distance between neighboring dots?
   Does the connecting line represent additional measurements? **No.**

**Exit ticket:** explain the two axes, the array orientation, why `P` is not
automatically an event label, and why successful reading does not establish
good signal quality. Changing a view does not change the recording itself.

**Next:** Lesson 3 asks which parts of a recording are trustworthy.
Later, a file you collect with NeuraDock can enter this same reading workflow,
once its acquisition settings and data permissions are documented.


In [ ]:
summary = {
    "lesson": 2, "mode": "replay", "synthetic": False,
    "source_sha256": SOURCE_SHA256,
    "provenance_basis": "supplied as a real recording by the data owner; not inferred from file format",
    "shape_channels_by_samples": [n_channels, n_samples],
    "declared_sample_rate_hz": FS, "nominal_duration_s": n_samples / FS,
    "channel_labels": CHANNEL_NAMES, "channel_mapping": "unconfirmed; column labels only",
    "amplitude_label": AMPLITUDE_LABEL, "preprocessing": "none",
    "packet_count": info["packet_count"], "samples_per_packet": info["samples_per_packet"],
    "malformed_row_count": len(info["malformed_rows"]),
    "packet_clock_summary": timing, "p_field_summary": p_summary,
    "experimental_events": "not inferred; requires a documented event source",
    "view": {"channel_index": CHANNEL_INDEX, "start_sample": start, "stop_sample_exclusive": stop},
    "figures": list(dict.fromkeys(FIGURES)),
    "quality_status": "not assessed in this reading lesson",
}
(OUTPUT_DIR / "summary.json").write_text(
    json.dumps(summary, indent=2, allow_nan=False), encoding="utf-8")
print("Saved three locally generated figures and summary.json in:", OUTPUT_DIR.relative_to(ROOT))
print("Keep the tracked notebook free of private outputs before sharing or committing.")
